# loralake on Colab

运行时类型选 **L4 GPU** 或更好的（A100 / G4 / H100）。T4 不支持 bf16，不适合训练。

数据放在 Google Drive 的 `MyDrive/loralake/` 下，会话结束后不会丢：

```
MyDrive/loralake/
├── config                 配置（第一次运行会从模板生成，填好 API key）
├── models/                ComfyUI 格式的 models 目录
│   ├── diffusion_models/  anima-base*.safetensors
│   ├── text_encoders/     qwen_3_06b*.safetensors
│   └── vae/               qwen_image_vae*.safetensors
├── datasets/              下载的数据集
└── output/                训练好的 LoRA
```

训练视图和 latent 缓存（`derived/`）放在本地 `/content`，读写快，会话结束即清空。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/loralake"
REPO = "/content/loralake"

## 安装

每个新会话都要运行一次，约几分钟。

In [ ]:
import os
if not os.path.isdir(REPO):
    !git clone --recurse-submodules https://github.com/Zhou-London/loralake.git {REPO}
else:
    !git -C {REPO} pull --recurse-submodules
!cd {REPO} && ./install.sh

## 配置

Drive 上的 `config` 是主配置；这里在它后面追加目录设置，生成仓库里实际使用的 `config`。

In [ ]:
import pathlib, shutil

drive_config = pathlib.Path(DRIVE, "config")
for sub in ("models/diffusion_models", "models/text_encoders", "models/vae", "datasets", "output"):
    pathlib.Path(DRIVE, sub).mkdir(parents=True, exist_ok=True)
if not drive_config.exists():
    shutil.copy(f"{REPO}/config.example", drive_config)
    print(f"已生成 {drive_config}，请在 Drive 中填写 API key 后重新运行本单元格")

overrides = f"""
# 以下由 colab.ipynb 追加，覆盖上面的同名项
COMFY_MODELS_PATH={DRIVE}/models
DATASET_DIR={DRIVE}/datasets
OUTPUT_DIR={DRIVE}/output
DERIVED_DIR=/content/derived
"""
pathlib.Path(REPO, "config").write_text(drive_config.read_text() + "\n" + overrides)
os.chdir(REPO)

# ! 命令的输出不是终端，rich 默认不显示动态进度条；强制开启（若输出错乱可删掉这三行）
os.environ["TTY_COMPATIBLE"] = "1"
os.environ["TTY_INTERACTIVE"] = "1"
os.environ["COLUMNS"] = "120"

## 使用

笔记本里没有交互输入，下载和训练都要加 `-y`。

In [ ]:
!llk -s 初音未来

In [ ]:
!llk -d hatsune_miku --limit 300 -y

In [ ]:
!llk -p hatsune_miku

In [ ]:
!llk -t hatsune_miku --trigger miku -y